<a href="https://colab.research.google.com/github/smkim0508/cos484-notes/blob/main/A2P1_Named_Entity_Recognition_HMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programming Problem 1: HMM for NER
Welcome to the programming portion of the assignment! Each assignment throughout the semester will have a written portion and a programming portion. We will be using [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb#recent=true), so if you have never used it before, take a quick look through this introduction: [Working with Google Colab](https://docs.google.com/document/d/1LlnXoOblXwW3YX-0yG_5seTXJsb3kRdMMRYqs8Qqum4/edit?usp=sharing).

### Writing Code
Look for the keyword "TODO" and fill in your code in the empty space.
Feel free to add and delete arguments in function signatures, but be careful that you might need to change them in function calls which are already present in the notebook.

### Data preprocessing

In this section we will write code to load data and build the dataset for Named Entity Recognition.

You may inspect the data first before writing the data preprocessing code by looking at the data file: https://princeton-nlp.github.io/cos484/assignments/a2/eng.train. Hints on processing the data:
- You may ignore the lines with "DOCSTART"
- Examples of NER tags include "O", "ORG", "MISC", and it's always in the same position in each line of the data.
- To process numbers more easily, you can replace all digits with 0's (to avoid out-of-vocab words)

You should end up with a list of sentences, where each sentence is represented with a list of words and tags.

Additional, you will want to support the following functions for later:
- Map words and tags to ids (integers)
- Handle unknown words in mapping

In [1]:
import re
from typing import Optional

class NERDataLoader():
    """
    Loads the NER dataset into a list of sentences.
    Each sentence is: (words: List[str], tags: List[str])

    - Skips lines containing "DOCSTART"
    - Tag is taken as the last whitespace-separated field on each line
    - Replaces digits with '0' to reduce OOV
    - Builds word/tag -> id maps from the training split
    - Unknown words map to <UNK>
    """

    PAD = "<PAD>"
    UNK = "<UNK>"

    def __init__(self, normalize_digits: bool = True):
        self.normalize_digits = normalize_digits
        self._digit_re = re.compile(r"\d")

        self.word_to_id: dict[str, int] = {self.PAD: 0, self.UNK: 1}
        self.tag_to_id: dict[str, int] = {self.PAD: 0}

        self.train: list[tuple[list[str], list[str]]] = []
        self.valid: list[tuple[list[str], list[str]]] = []

    def load(self, path: str, *, split: str) -> None:
        sentences = self._read_file(path)
        if split == "train":
            self.train = sentences
        elif split in ("valid", "val", "dev"):
            self.valid = sentences
        else:
            raise ValueError(f"split must be 'train' or 'valid/val/dev', got {split!r}")

    def build_id_maps(self) -> None:
        if not self.train:
            raise ValueError("Load training data first (split='train') before building id maps.")

        for words, tags in self.train:
            for w in words:
                if w not in self.word_to_id:
                    self.word_to_id[w] = len(self.word_to_id)
            for t in tags:
                if t not in self.tag_to_id:
                    self.tag_to_id[t] = len(self.tag_to_id)

    def encode(self, words: list[str], tags: Optional[list[str]] = None):
        w_ids = [self.word_to_id.get(w, self.word_to_id[self.UNK]) for w in words]
        if tags is None:
            return w_ids
        t_ids = [self.tag_to_id[t] for t in tags]
        return w_ids, t_ids

    def _read_file(self, path: str) -> list[tuple[list[str], list[str]]]:
        sentences: list[tuple[list[str], list[str]]] = []
        cur_words: list[str] = []
        cur_tags: list[str] = []

        def flush():
            nonlocal cur_words, cur_tags
            if cur_words:
                sentences.append((cur_words, cur_tags))
                cur_words, cur_tags = [], []

        with open(path, "r", encoding="utf-8") as f:
            for raw in f:
                line = raw.strip()

                if not line:
                    flush()
                    continue

                if "DOCSTART" in line:
                    continue

                parts = line.split()
                word = parts[0]
                tag = parts[-1]

                if self.normalize_digits:
                    word = self._digit_re.sub("0", word)

                cur_words.append(word)
                cur_tags.append(tag)

        flush()
        return sentences

In [2]:
# download the provided data

import urllib.request

url = "https://princeton-nlp.github.io/cos484/assignments/a2/eng.train"
out_path = "eng.train"

urllib.request.urlretrieve(url, out_path)
print("saved to", out_path)

saved to eng.train


In [6]:
# verify that the loader is working properly

loader = NERDataLoader(normalize_digits=True)
loader.load("eng.train", split="train")
loader.build_id_maps()

print("# sentences:", len(loader.train))
print("# word types:", len(loader.word_to_id))
print("# tag types:", len(loader.tag_to_id))
print("tags:", sorted([t for t in loader.tag_to_id.keys() if t != loader.PAD]))

# sentences: 14041
# word types: 20102
# tag types: 6
tags: ['LOC', 'MISC', 'O', 'ORG', 'PER']


### Hidden Markov Model
In this section, we will implement a bigram hidden markov model (HMM) that could perform two types of decoding: greedy decoding and viterbi decoding.

Specifically, you should include the following functionalities:
- Initialize the HMM given the word and tag mappings.
- Train the HMM with a given corpus
- Greedy decoding: given a single sentence, output its tags with greedy algorithm
- Viterbi decoding: given a single sentence, output its tags using Viterbi

You may refer to the lecture notes for more details on the HMM and the decoding algorithms.

In [ ]:
import math
from typing import Optional

NEG_INF = -1e30


class BigramHMM():
    """
    Bigram HMM for NER.

    Learns:
      - transition probs: p(tag_t | tag_{t-1})
      - emission probs:   p(word_t | tag_t)

    Decoding:
      - greedy
      - viterbi

    NOTE:
      - Uses <START> and <STOP> internally as tags (not emitted).
      - Uses add-alpha smoothing.
      - Works in log-space.
    """

    START = "<START>"
    STOP = "<STOP>"

    def __init__(
        self,
        word_to_id: dict[str, int],
        tag_to_id: dict[str, int],
        *,
        unk_token: str = "<UNK>",
        alpha_trans: float = 1e-3,
        alpha_emit: float = 1e-3,
    ):
        self.word_to_id = dict(word_to_id)
        self.tag_to_id = dict(tag_to_id)
        self.unk_token = unk_token
        self.alpha_trans = alpha_trans
        self.alpha_emit = alpha_emit

        # ensure START/STOP exist
        if self.START not in self.tag_to_id:
            self.tag_to_id[self.START] = len(self.tag_to_id)
        if self.STOP not in self.tag_to_id:
            self.tag_to_id[self.STOP] = len(self.tag_to_id)

        self.id_to_tag = {i: t for t, i in self.tag_to_id.items()}

        self.start_id = self.tag_to_id[self.START]
        self.stop_id = self.tag_to_id[self.STOP]

        self.T = len(self.tag_to_id)
        self.V = len(self.word_to_id)

        self.logA: Optional[list[list[float]]] = None  # [prev_tag][tag]
        self.logB: Optional[list[list[float]]] = None  # [tag][word]

    def fit(self, train: list[tuple[list[str], list[str]]]) -> None:
        # counts
        trans = [[0] * self.T for _ in range(self.T)]
        trans_den = [0] * self.T

        emit = [[0] * self.V for _ in range(self.T)]
        emit_den = [0] * self.T

        def wid(w: str) -> int:
            return self.word_to_id.get(w, self.word_to_id[self.unk_token])

        def tid(t: str) -> int:
            return self.tag_to_id[t]

        for words, tags in train:
            prev = self.start_id
            for w, t in zip(words, tags):
                y = tid(t)
                x = wid(w)

                trans[prev][y] += 1
                trans_den[prev] += 1

                emit[y][x] += 1
                emit_den[y] += 1

                prev = y

            # STOP transition
            trans[prev][self.stop_id] += 1
            trans_den[prev] += 1

        # log transitions
        aT = self.alpha_trans
        self.logA = [[0.0] * self.T for _ in range(self.T)]
        for prev in range(self.T):
            denom = trans_den[prev] + aT * self.T
            for y in range(self.T):
                p = (trans[prev][y] + aT) / denom
                self.logA[prev][y] = math.log(p)

        # log emissions
        aE = self.alpha_emit
        self.logB = [[0.0] * self.V for _ in range(self.T)]
        for y in range(self.T):
            denom = emit_den[y] + aE * self.V
            if denom == 0:
                # START/STOP (or unseen tag) don't emit; define uniform for safety
                uni = -math.log(self.V)
                for x in range(self.V):
                    self.logB[y][x] = uni
            else:
                for x in range(self.V):
                    p = (emit[y][x] + aE) / denom
                    self.logB[y][x] = math.log(p)

    def greedy_decode(self, words: list[str]) -> list[int]:
        assert self.logA is not None and self.logB is not None

        def wid(w: str) -> int:
            return self.word_to_id.get(w, self.word_to_id[self.unk_token])

        path: list[int] = []
        prev = self.start_id

        for w in words:
            x = wid(w)
            best_y = -1
            best_score = NEG_INF

            for y in range(self.T):
                if y == self.start_id or y == self.stop_id:
                    continue
                score = self.logA[prev][y] + self.logB[y][x]
                if score > best_score:
                    best_score = score
                    best_y = y

            path.append(best_y)
            prev = best_y

        return path

    def viterbi_decode(self, words: list[str]) -> list[int]:
        assert self.logA is not None and self.logB is not None
        n = len(words)
        if n == 0:
            return []

        def wid(w: str) -> int:
            return self.word_to_id.get(w, self.word_to_id[self.unk_token])

        dp = [[NEG_INF] * self.T for _ in range(n)]
        bp = [[-1] * self.T for _ in range(n)]

        # init from START
        x0 = wid(words[0])
        for y in range(self.T):
            if y == self.start_id or y == self.stop_id:
                continue
            dp[0][y] = self.logA[self.start_id][y] + self.logB[y][x0]
            bp[0][y] = self.start_id

        # DP
        for t in range(1, n):
            x = wid(words[t])
            for y in range(self.T):
                if y == self.start_id or y == self.stop_id:
                    continue

                best_prev = -1
                best_val = NEG_INF
                for prev in range(self.T):
                    val = dp[t - 1][prev] + self.logA[prev][y] + self.logB[y][x]
                    if val > best_val:
                        best_val = val
                        best_prev = prev

                dp[t][y] = best_val
                bp[t][y] = best_prev

        # end with STOP
        best_last = -1
        best_score = NEG_INF
        for y in range(self.T):
            if y == self.start_id or y == self.stop_id:
                continue
            score = dp[n - 1][y] + self.logA[y][self.stop_id]
            if score > best_score:
                best_score = score
                best_last = y

        # backtrack
        path = [best_last]
        for t in range(n - 1, 0, -1):
            path.append(bp[t][path[-1]])
        path.reverse()
        return path

    def decode_ids(self, tag_ids: list[int]) -> list[str]:
        return [self.id_to_tag[i] for i in tag_ids]

### Train and evaluate HMMs.

In this section, you will implement the logic for training and evaluating the HMMs:
- Train the model by calling the functions/classes you implemented above,
- Evaluate the trained model on the training and evaluation set by calculating the accuracy of the predicted tags.
- Compute the confusion matrix and F1 score of the predictions.

### Experiments

#### Load data

Download and load the data from the following links

https://princeton-nlp.github.io/cos484/assignments/a2/eng.train

https://princeton-nlp.github.io/cos484/assignments/a2/eng.val

Then load the data using what you implemented

#### Experiment with an HMM with greedy decoding

**(a) Which pair of tags does the model have most difficulty separating according to the confusion matrix of the validation set?**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)


#### Experiment with an HMM with viterbi decoding

**(b) What major differences do you observe compared to the matrix in (a)**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)


# LLM Prompts

If you used an AI tool to complete any part of this assignment, please paste all prompts you used to produce your final code/responses in the box below and answer the following reflection question.

Prompts Used:
*   
*   



**Reflection: What parts of the AI generated output required modification or improvement? Describe the feedback you gave the tool to produce your final output or any changes you had to make on your own.**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)